# XLA Final — Object Detection Training
ResNet18 + FPN + anchor-based one-stage detector


In [ ]:
import subprocess, os, sys

# Verify GPU
subprocess.run(['nvidia-smi'], check=True)
print('Python:', sys.version)

In [ ]:
REPO_URL = 'https://github.com/Dung205789/XLA_final.git'
REPO_DIR = '/kaggle/working/XLA_final'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Link the Kaggle-mounted dataset to ./public/
# Dataset 'dzungngo179/xla-final-public' is mounted at /kaggle/input/xla-final-public/
DATASET_PATH = '/kaggle/input/xla-final-public'

if not os.path.exists(f'{REPO_DIR}/public'):
    os.symlink(DATASET_PATH, f'{REPO_DIR}/public')
    print('Symlinked public/ →', DATASET_PATH)
else:
    print('public/ already exists')

# Verify dataset structure
for p in ['annotations/train.json', 'annotations/val.json', 'train/images', 'val/images']:
    full = f'{REPO_DIR}/public/{p}'
    print(f'  {p}: {"OK" if os.path.exists(full) else "MISSING"}')

In [ ]:
# Install requirements
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Requirements installed.')

In [ ]:
# Quick sanity check: count images
import json
with open('public/annotations/train.json') as f:
    td = json.load(f)
with open('public/annotations/val.json') as f:
    vd = json.load(f)
print(f'Train: {len(td["images"])} images, {len(td["annotations"])} boxes')
print(f'Val  : {len(vd["images"])} images, {len(vd["annotations"])} boxes')
print(f'Classes: {td["classes"]}')

In [ ]:
# Run training
# T4/P100 GPU on Kaggle: batch_size=16 fits comfortably
cmd = [
    sys.executable, 'train.py',
    '--train_data',     'public/annotations/train.json',
    '--val_data',       'public/annotations/val.json',
    '--image_dir',      'public/train/images',
    '--val_image_dir',  'public/val/images',
    '--checkpoint_dir', 'models/',
    '--epochs',         '40',
    '--batch_size',     '16',
    '--lr',             '3e-4',
    '--input_size',     '512',
    '--workers',        '2',
    '--freeze_epochs',  '3',
    '--conf_thresh',    '0.05',
    '--nms_thresh',     '0.5',
]

print('Starting training …')
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=False)
print('Return code:', result.returncode)

In [ ]:
# Evaluate best checkpoint on validation set
subprocess.run([
    sys.executable, 'predict.py',
    '--image_dir',   'public/val/images',
    '--output',      'val_predictions.json',
    '--checkpoint',  'models/best.pth',
    '--anchor_cfg',  'models/anchors.json',
    '--conf_thresh', '0.05',
    '--nms_thresh',  '0.5',
], check=True)

subprocess.run([
    sys.executable, 'public/tools/evaluate_predictions.py',
    '--ground_truth', 'public/annotations/val.json',
    '--predictions',  'val_predictions.json',
    '--output',       'val_score.json',
], check=True)

with open('val_score.json') as f:
    score = json.load(f)

print('='*60)
print(f'mAP@0.5 = {score["mAP@0.5"]:.4f}')
print(f'micro precision = {score["micro_precision"]:.4f}')
print(f'micro recall    = {score["micro_recall"]:.4f}')
print('Per-class AP:')
for cls, info in score['per_class'].items():
    print(f'  {cls:8s}  AP={info["ap"]:.4f}  recall={info["recall"]:.4f}  precision={info["precision"]:.4f}')
print('='*60)

In [ ]:
# Copy best.pth to /kaggle/working/ for download
import shutil
shutil.copy('models/best.pth', '/kaggle/working/best.pth')
shutil.copy('models/anchors.json', '/kaggle/working/anchors.json')
shutil.copy('val_score.json', '/kaggle/working/val_score.json')
print('Files copied to /kaggle/working/')
print('best.pth size:', os.path.getsize('/kaggle/working/best.pth') // 1024 // 1024, 'MB')

In [ ]:
# Optional: tune confidence threshold on val
import json

def eval_thresh(conf_t, nms_t=0.5):
    subprocess.run([
        sys.executable, 'predict.py',
        '--image_dir', 'public/val/images',
        '--output', f'/tmp/pred_{conf_t:.3f}.json',
        '--checkpoint', 'models/best.pth',
        '--anchor_cfg', 'models/anchors.json',
        '--conf_thresh', str(conf_t),
        '--nms_thresh', str(nms_t),
    ], check=True, capture_output=True)
    r = subprocess.run([
        sys.executable, 'public/tools/evaluate_predictions.py',
        '--ground_truth', 'public/annotations/val.json',
        '--predictions', f'/tmp/pred_{conf_t:.3f}.json',
        '--output', '/tmp/score.json',
    ], capture_output=True, text=True)
    s = json.loads(r.stdout)
    return s.get('mAP@0.5', 0)

print('Threshold tuning:')
for ct in [0.03, 0.05, 0.07, 0.10, 0.15]:
    m = eval_thresh(ct)
    print(f'  conf={ct:.3f}  mAP={m:.4f}')